# Experiment 9: Personalized AI Assistant using LLM + RAG + optional LoRA
This notebook compares a base model response with a document-grounded RAG response. Optional fine-tuning is provided as a separate script and requires a prepared dataset and compatible GPU.

In [ ]:
%pip install -q transformers accelerate sentence-transformers faiss-cpu pypdf

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## 1. Load the assistant code
Upload `assistant_rag.py` to Colab, or upload/extract the full ZIP and change to its folder.

In [ ]:
from assistant_rag import RAGAssistant
assistant = RAGAssistant('Qwen/Qwen2.5-0.5B-Instruct')
print('Assistant loaded.')

## 2. Upload a document
Use a PDF, TXT or Markdown document you are allowed to process. Do not use confidential information.

In [ ]:
from google.colab import files
uploaded = files.upload()
doc_paths = list(uploaded.keys())
print('Uploaded:', doc_paths)

In [ ]:
for path in doc_paths:
    if path.lower().endswith(('.pdf', '.txt', '.md')):
        print(path, 'chunks added:', assistant.add_document(path))

## 3. Compare base model response and RAG response

In [ ]:
question = 'Summarize the main points in my uploaded document.'
base_answer, _ = assistant.answer(question, use_rag=False)
rag_answer, retrieved_chunks = assistant.answer(question, use_rag=True)
print('BASE MODEL ANSWER:\n', base_answer)
print('\nRAG ANSWER:\n', rag_answer)
print('\nRETRIEVED CONTEXT:\n', '\n---\n'.join(retrieved_chunks))

## 4. Try your own question

In [ ]:
question = 'What does the document say about its main topic?'
answer, chunks = assistant.answer(question, use_rag=True)
print(answer)
print('\nSources used:\n', '\n---\n'.join(chunks))

## 5. Optional LoRA fine-tuning
Fine-tuning is not required for the RAG demo. Prepare `data/train.jsonl` with one JSON object per line containing a `text` field. Upload the ZIP/project and run `python finetune_lora.py --dataset data/train.jsonl` only when a compatible GPU and sufficient memory are available. Compare base, RAG and fine-tuned outputs using the same questions.